# Stage 3 DPD Safe-Zone — Phase B: notebook setup

**ПРОСТЫМ ЯЗЫКОМ:** Загружаем 4 сырых CSV из SQL-выгрузки (`stage3_safezone_rolling_extract.sql`), строим для каждого из 12 отчётных месяцев 6-месячное окно DPD по займу, и определяем состояние реструктуризации по каждому месяцу окна — **три состояния, не «да/нет»**: `active` (каникулы действовали), `not_active` (точно не действовали), `unknown` (событие есть, но дат каникул в источнике нет — судить не можем). Отсюда `restr_active_pct` и `restr_unknown_pct` по каждому займу. Это подготовка данных для Фазы C (симуляция порога) — сама симуляция здесь ещё не реализована.

See [`docs/analysis/stage3_safezone_plan.md`](../docs/analysis/stage3_safezone_plan.md) for the full plan (Phases A–E).

**Before running:** point `RAW_DATA_DIR` below at your exported CSVs and confirm the filenames in `FILES` match what you actually exported — the diagnostic cell right after lists what's actually in the folder.

**What to check before moving to Phase C:** the row-count sanity check, then `mean_unknown_pct` and `pct_event_dated`. If the unknown share is material, the restructured-vs-not split in Phase C rests on a denominator of unknown quality and has to be reported that way — not quietly folded into "not restructured."

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DATA_DIR = Path(r"C:\project_mz\surau\DPDRelaxing\raw_data")

# Adjust these to your actual exported filenames if they differ.
FILES = {
    "report_dates": "report_dates.csv",
    "stage3_pool": "stage3_pool.csv",
    "dpd_panel": "dpd_panel.csv",
    "restructuring_events": "restructuring_events.csv",
}

LAST_ASOF = pd.Timestamp("2026-07-01")
LOOKBACK_MONTHS = 6

In [ ]:
print("CSV files found in RAW_DATA_DIR:")
for f in sorted(RAW_DATA_DIR.glob("*.csv")):
    print(" ", f.name)

## Restore column headers

**ПРОСТЫМ ЯЗЫКОМ:** SSMS-экспорт в CSV теряет заголовок колонок, но порядок колонок сохраняется таким, как в SELECT в `stage3_safezone_rolling_extract.sql`. Ниже — четыре списка имён в точном порядке запроса (для `stage3_pool` — все 69 колонок `CL_PORTFOLIO_2` в табличном порядке, плюс `portfolio_label`/`portfolio_asof`) и функция-загрузчик, которая присваивает имена, проверяет число колонок и на всякий случай снимает заголовок, если он всё-таки просочился.

If the SQL result set changes, update the matching `COLUMNS_*` list here — the count check below will fail loudly rather than silently misaligning columns.

In [ ]:
import csv

# stage3_safezone_rolling_extract.sql §0 -- SELECT * FROM ##SAFEZONE_REPORT_DATES
COLUMNS_REPORT_DATES = ["asof_date", "months_back", "portfolio_label"]

# §1 -- SELECT a.*, rd.portfolio_label, rd.asof_date AS portfolio_asof FROM CL_PORTFOLIO_2 a ...
# a.* is every CL_PORTFOLIO_2 column in table order (confirmed via INFORMATION_SCHEMA.COLUMNS).
COLUMNS_STAGE3_POOL = [
    "contract_number", "status", "granting_date", "first_pmt_date", "outstanding",
    "outstanding_overdue", "interest", "overdue_interest",
    "accrued_interest_on_overdue_outstanding", "overdue_interest_on_overdue_outstanding",
    "penalties", "termination_commissions", "loan_servicing_commissions",
    "overdue_commissions_income", "overdue_days_principal", "overdue_days_interest",
    "dpd", "category", "percent", "ifrs", "provisions_calculated", "loan_purpose",
    "subproduct", "discount_1434", "discount_1435", "discount_1773", "fees_1860",
    "attribute_marker", "date_of_attribute_marker", "filial", "merchant_city",
    "LoanDuration", "FKLogin", "IIN", "CARBRAND", "teh_overdfraft",
    "overdue_commissions", "overdue_penalties", "od", "balance", "product", "basket",
    "90+", "90+sum", "fin_date_short", "date", "tag", "tag_1", "rezident", "valuta",
    "fiziki/yuriki", "IFRS1877", "IFRS1845", "maxDPD", "ifrs18770_DEB",
    "ifrs18771_WTRAF_i_PENII", "OD_percent", "tarif", "discount_1774", "discount_1775",
    "discount_1784", "balance_with_discount", "provisions_total", "DISCOUNT_1484",
    "DISCOUNT_14341", "DISCOUNT_14342", "DISCOUNT_1485", "DISCOUNT_17731",
    "DISCOUNT_17732",
    # appended by the extract query, not part of CL_PORTFOLIO_2 itself:
    "portfolio_label", "portfolio_asof",
]

# §2 -- SELECT p.contract_number, p.[date] AS snap_date, p.[dpd], p.[category], p.[balance],
#         p.[balance_with_discount], p.[provisions_total], p.[tag]
COLUMNS_DPD_PANEL = [
    "contract_number", "snap_date", "dpd", "category", "balance",
    "balance_with_discount", "provisions_total", "tag",
]

# §3 -- SELECT r.* FROM [Dictionaries].[risk_analytics].[restructuring_v2] r ...
COLUMNS_RESTRUCTURING_EVENTS = [
    "dlcr_gid", "dlcr$source", "loan_id", "restructuring_date", "new_interest_rate",
    "days_past_due_at_restructuring", "new_maturity_date", "financial_deterioration_flag",
    "payment_deferral", "canc_date", "grace_od_begin_date", "grace_int_begin_date",
    "grace_od_end_date", "grace_int_end_date", "report_date",
]


def read_headerless_csv(
    path: Path, columns: list, parse_dates: list = None
) -> pd.DataFrame:
    """Read a CSV whose header row was dropped on export, restoring names from the SQL
    SELECT order. Also tolerates a header row that slipped through anyway (detected by
    comparing the first row to `columns` and skipped), and fails loudly on a column-count
    mismatch instead of silently misaligning data under the wrong names."""
    with open(path, newline="", encoding="utf-8-sig") as f:
        first_row = next(csv.reader(f))
    skip = 1 if [v.strip() for v in first_row] == columns else 0

    df = pd.read_csv(path, header=None, skiprows=skip, names=columns)
    if len(df.columns) != len(columns):
        raise ValueError(
            f"{path.name}: expected {len(columns)} columns (SQL SELECT order), found "
            f"{len(df.columns)} in the file -- update the matching COLUMNS_* list to match "
            "stage3_safezone_rolling_extract.sql."
        )
    if parse_dates:
        for col in parse_dates:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

## Load raw extracts

Matches the four result sets from `sql/stage3_safezone_rolling_extract.sql` §0/§1/§2/§3, with headers restored by `read_headerless_csv`.

In [ ]:
report_dates = read_headerless_csv(
    RAW_DATA_DIR / FILES["report_dates"], COLUMNS_REPORT_DATES, parse_dates=["asof_date"]
)

stage3_pool = read_headerless_csv(
    RAW_DATA_DIR / FILES["stage3_pool"],
    COLUMNS_STAGE3_POOL,
    parse_dates=["date", "portfolio_asof"],
)

dpd_panel = read_headerless_csv(
    RAW_DATA_DIR / FILES["dpd_panel"], COLUMNS_DPD_PANEL, parse_dates=["snap_date"]
)

restructuring_events = read_headerless_csv(
    RAW_DATA_DIR / FILES["restructuring_events"],
    COLUMNS_RESTRUCTURING_EVENTS,
    parse_dates=[
        "restructuring_date",
        "new_maturity_date",
        "canc_date",
        "grace_od_begin_date",
        "grace_od_end_date",
        "grace_int_begin_date",
        "grace_int_end_date",
        "report_date",
    ],
)

## Sanity check — row counts against what SQL Server reported (23.07.2026)

In [ ]:
expected = {
    "report_dates": 12,
    "stage3_pool": 481_818,
    "dpd_panel": 1_080_891,
    "restructuring_events": 91_679,
}
actual = {
    "report_dates": len(report_dates),
    "stage3_pool": len(stage3_pool),
    "dpd_panel": len(dpd_panel),
    "restructuring_events": len(restructuring_events),
}
for name, expected_count in expected.items():
    got = actual[name]
    flag = "OK" if got == expected_count else "CHECK — differs from the SQL-side count"
    print(f"{name:22s} expected {expected_count:>10,}  got {got:>10,}  [{flag}]")

## Per-portfolio 6-month lookback window

For a given `portfolio_asof`, slice `dpd_panel` down to the `LOOKBACK_MONTHS` ending at that date (inclusive), and label each row with a `month_offset` (0 = the portfolio's own month, negative = further back).

In [ ]:
def build_lookback_dpd(
    dpd_panel: pd.DataFrame, portfolio_asof: pd.Timestamp, lookback_months: int = LOOKBACK_MONTHS
) -> pd.DataFrame:
    """DPD/category rows for the lookback window ending at portfolio_asof (inclusive)."""
    window_start = portfolio_asof - pd.DateOffset(months=lookback_months - 1)
    window = dpd_panel[
        (dpd_panel["snap_date"] >= window_start) & (dpd_panel["snap_date"] <= portfolio_asof)
    ].copy()
    window["month_offset"] = (
        (window["snap_date"].dt.year - portfolio_asof.year) * 12
        + (window["snap_date"].dt.month - portfolio_asof.month)
    )
    return window

## Restructuring state per month — `active` / `not_active` / `unknown`

**ПРОСТЫМ ЯЗЫКОМ:** три состояния вместо «да/нет». Займ считается **под реструктуризацией** (`active`), если на дату среза есть уже начавшееся и не отменённое событие, и срез попадает внутрь окна каникул — по основному долгу (`grace_od_*`) или по вознаграждению (`grace_int_*`). Если событие есть, но **даты каникул в источнике не заполнены** — это `unknown`: реструктуризация была, покрывала ли она этот месяц — мы не знаем. `not_active` ставим только когда действительно можем это утверждать.

Why three states and not a boolean: an event with **no grace dates populated at all** cannot be judged either way. The previous boolean folded that case into `False`, which reports a genuinely restructured loan as "no payment holiday" — and quietly biases every restructured-vs-not comparison in Phase C, on a denominator of unknown quality.

| State | Meaning |
|---|---|
| `active` | a qualifying event's grace window covers this snapshot |
| `unknown` | a qualifying event exists but carries no usable grace dates to check against |
| `not_active` | no qualifying event at all, **or** every qualifying event has dates and this snapshot falls outside all of them |

"Qualifying" = `restructuring_date <= snap_date` and not cancelled by then (`canc_date` null or later). A loan can match several events — states aggregate with `any()` across all of them, since an older event's grace window can still be running. One *undated* qualifying event is enough to make the month `unknown`, even if a sibling event has dates that don't cover it: we cannot rule coverage out.

Read `restr_unknown_pct` as a **data-quality** reading, not a risk reading — a high value means the source can't answer for that loan, not that the loan is safe.

In [ ]:
GRACE_PAIRS = [
    ("grace_od_begin_date", "grace_od_end_date"),      # principal holiday
    ("grace_int_begin_date", "grace_int_end_date"),    # interest holiday
]


def _has_grace_dates(events: pd.DataFrame) -> pd.Series:
    """Per event row: is at least one grace pair fully populated (both ends)?
    A half-populated pair is unusable — an open-ended window can't be tested."""
    present = pd.Series(False, index=events.index)
    for begin_col, end_col in GRACE_PAIRS:
        present = present | (events[begin_col].notna() & events[end_col].notna())
    return present


def classify_restructuring(
    dpd_window: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """One row per (contract_number, snap_date) with restr_state in
    {'active', 'not_active', 'unknown'} — see the markdown above for the rules."""
    events = restructuring_events.rename(columns={"loan_id": "contract_number"})
    merged = dpd_window.merge(events, on="contract_number", how="left")

    # Started by this snapshot and not cancelled before it. NaT comparisons are False,
    # so contracts with no event at all fall out here and end up 'not_active'.
    qualifies = (merged["restructuring_date"] <= merged["snap_date"]) & (
        merged["canc_date"].isna() | (merged["canc_date"] > merged["snap_date"])
    )

    dated = _has_grace_dates(merged)
    in_grace = pd.Series(False, index=merged.index)
    for begin_col, end_col in GRACE_PAIRS:
        in_grace = in_grace | (
            merged[begin_col].notna()
            & merged[end_col].notna()
            & (merged["snap_date"] >= merged[begin_col])
            & (merged["snap_date"] <= merged[end_col])
        )

    merged["_active"] = qualifies & in_grace
    merged["_undated"] = qualifies & ~dated

    per_month = (
        merged.groupby(["contract_number", "snap_date"])[["_active", "_undated"]]
        .any()
        .reset_index()
    )
    # Priority: a confirmed covering window wins; otherwise any undated qualifying
    # event makes the month unanswerable; only then can we assert not_active.
    per_month["restr_state"] = np.where(
        per_month["_active"],
        "active",
        np.where(per_month["_undated"], "unknown", "not_active"),
    )
    return per_month[["contract_number", "snap_date", "restr_state"]]


def compute_restr_state_pct(
    dpd_window: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """Share of each contract's OBSERVED months in the window per state.
    Columns: restr_active_pct / restr_unknown_pct / restr_not_active_pct (sum to 1.0).

    Denominator is months actually present in the panel, not LOOKBACK_MONTHS — a loan
    with a missing snapshot must not be diluted toward zero as if that month were clean.
    """
    states = classify_restructuring(dpd_window, restructuring_events)
    counts = (
        states.pivot_table(
            index="contract_number",
            columns="restr_state",
            values="snap_date",
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=["active", "unknown", "not_active"], fill_value=0)
    )
    pct = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
    pct.columns = ["restr_active_pct", "restr_unknown_pct", "restr_not_active_pct"]
    return pct.reset_index()

## Build the per-month state shares for all 12 portfolio months

Two matrices (loan × portfolio month): `restr_active_pct` — the risk signal Phase C splits on — and `restr_unknown_pct` — how much of that split is guesswork. Look at the summary underneath before trusting either: a month with a high mean unknown share cannot support a "restructured vs not" comparison at all.

In [ ]:
restr_state_pct_by_portfolio = {}

for _, row in report_dates.iterrows():
    window = build_lookback_dpd(dpd_panel, row["asof_date"])
    restr_state_pct_by_portfolio[row["portfolio_label"]] = compute_restr_state_pct(
        window, restructuring_events
    ).set_index("contract_number")

restr_active_pct = pd.concat(
    {k: v["restr_active_pct"] for k, v in restr_state_pct_by_portfolio.items()}, axis=1
)
restr_unknown_pct = pd.concat(
    {k: v["restr_unknown_pct"] for k, v in restr_state_pct_by_portfolio.items()}, axis=1
)

# Per-month readout. mean_unknown_pct is the one to watch: it caps how much weight the
# restructured-vs-not split in Phase C can carry for that month.
pd.DataFrame(
    {
        "mean_active_pct": restr_active_pct.mean().round(3),
        "mean_unknown_pct": restr_unknown_pct.mean().round(3),
        "loans_with_any_unknown_month": (restr_unknown_pct > 0).sum(),
        "loans_in_window": restr_active_pct.notna().sum(),
    }
)

## Transparency metric — coverage *and* usability of the restructuring source

**ПРОСТЫМ ЯЗЫКОМ:** два разных вопроса, и оба нужны. Первый — у какой доли пула вообще есть событие реструктуризации. Второй — у какой доли этих событий заполнены даты каникул, без которых проверка на «активна ли она в этом месяце» не работает. Высокий `pct_with_event` при низком `pct_event_dated` означает: реструктуризации были, но когда именно они действовали — источник не знает.

Phase B of the plan asks for "% of the population with a restructuring event **defined vs. not**". `pct_with_event` alone answers the weaker question — a loan can have an event on record and still be unanswerable. Both columns together are the actual transparency metric, and `pct_event_dated` is what bounds the credibility of the Phase C split.

In [ ]:
def restructuring_coverage_summary(
    stage3_pool: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """Per portfolio month: how many Stage 3 loans have a restructuring event at all,
    and how many have one with usable grace dates.

    pct_event_dated is deliberately a share OF LOANS WITH AN EVENT, not of the whole
    pool — it answers "when we do have an event, can we use it?", which is the question
    that governs whether restr_active_pct means anything for that month.
    """
    events = restructuring_events.assign(
        has_grace_dates=_has_grace_dates(restructuring_events)
    )
    with_event = set(events["loan_id"].unique())
    with_dated_event = set(events.loc[events["has_grace_dates"], "loan_id"].unique())

    tagged = stage3_pool.assign(
        has_restructuring_event=lambda d: d["contract_number"].isin(with_event),
        has_dated_event=lambda d: d["contract_number"].isin(with_dated_event),
    )
    summary = tagged.groupby("portfolio_label").agg(
        total_loans=("contract_number", "count"),
        with_event=("has_restructuring_event", "sum"),
        with_dated_event=("has_dated_event", "sum"),
    )
    summary["pct_with_event"] = (
        summary["with_event"] / summary["total_loans"] * 100
    ).round(1)
    summary["pct_event_dated"] = (
        summary["with_dated_event"] / summary["with_event"].replace(0, np.nan) * 100
    ).round(1)
    return summary.sort_index()


event_fill = _has_grace_dates(restructuring_events)
print(
    f"Event-level grace-date fill rate: {event_fill.mean() * 100:.1f}% "
    f"({int(event_fill.sum()):,} of {len(event_fill):,} event rows carry a usable pair)"
)

coverage = restructuring_coverage_summary(stage3_pool, restructuring_events)
coverage

## Next: Phase C (not yet built here)

This notebook covers Phase B of `stage3_safezone_plan.md` — loading the raw extracts, building the 6-month lookback windows, the three-state restructuring classification (`restr_active_pct` / `restr_unknown_pct`) and the coverage/usability metric.

**Run this first and confirm three things before Phase C:**

1. **Row counts** match the SQL-side numbers (the sanity-check cell).
2. **`mean_unknown_pct`** per portfolio month — how much of the restructuring signal is unanswerable. This bounds what the restructured-vs-not split in Phase C can claim.
3. **`pct_event_dated`** — of the loans that have a restructuring event, how many carry usable grace dates.

**Still open before Phase C can be written** (see the plan's "Locked methodology"):

- **A fixed re-default horizon K.** `@LastAsOf` equals the newest portfolio date, so forward runway ranges from 11 months (08.2025) to zero (07.2026). Scanning "up to the latest available report date" makes the 12 columns non-comparable and pushes the apparent re-default rate down toward the recent end — an elbow picked off that curve is a censoring artifact. Proposal on the table: K=6, main matrix restricted to cohorts with full runway (08.2025–01.2026), the rest reported separately and labelled incomplete.
- **`censoring_events.csv`** from `scripts/writeoff_restoration_scan.py`, run against the real archive. Loans that left the panel via sale/write-off/forgiveness must be censored, not counted as clean survivors.

Phase C (threshold × re-default matrix over n ∈ {0, 3, 7, …, 30}) and Phase D (H1 vs H2) build on top of what's here.